<h2>Modelling: Future Price Direction

The ML aspects of this project will be divided into two separate parts: 
- Predicting future price direction and,
- Predicting future price volatilty

It's very dificult predicting exact future price but if future direction and volatility can be predicted, an agent can take this information and size a position accordingly.

I want to create a model which takes in a sequence of prices and outputs a probability of the next price being higher or lower than the last (or the average) of the sequence.

One concern I have is if the day to day prices are too noisy that the next predicted value may not be particularly accurate i.e. it may also be noise. The model may have a low conviction in a decisive price movement. This could be overcome potentially by having the model take in some data and then say which the direction the price may be 1 week / 1 month from now, with an associated probability. This combined with a predicted volatility (to be modelled separately) may contain better information.

Furthermore, just price quotes may not be enough. It may be necessary to add price indicators such as RSI, MACD, and moving averages.

In [1]:
import sys
from pathlib import Path

PROJ_ROOT = Path().resolve().parents[0]
project_root = str(PROJ_ROOT)
sys.path.insert(0, project_root)

In [2]:
DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_PATH = RAW_DATA_DIR / "btcusd_data.pkl"
print(RAW_DATA_PATH)

C:\Users\cochr\Data Science Projects\vlXsnPPgKKnWqEJB\data\raw\btcusd_data.pkl


In [3]:
import pickle
import numpy as np
import pandas as pd

<h3> Logistic Regression

In [4]:
# Only load pickle files from a trusted source; unpickling can execute code.
with RAW_DATA_PATH.open("rb") as file:
    raw_data = pickle.load(file)

raw_data.head()

Price,Date,Close,High,Low,Open,Volume
0,2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800
1,2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200
2,2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700
3,2014-09-20,408.903992,423.295990,389.882996,394.673004,36863600
4,2014-09-21,398.821014,412.425995,393.181000,408.084991,26580100


We will take 'Close' as the current price. From here, we need to create columns that offset the Close value to a given interval in the future e.g. 1 day, 5 day, 10 day, then return a binary value saying if it is higher (1) or lower (0) than the Close of the current day.